In [84]:
import pandas as pd
import numpy as np
df = pd.read_csv("TankFrontSide45-50.csv")
df.head()
print(df.shape)

(43, 23)


In [85]:
df[df['median_m'].between(0.45,0.5)]

,timestamp,bbox_x1,bbox_y1,bbox_x2,bbox_y2,conf,class,center_m,median_m,pose_x,...,ori_z,ori_w,linear_x,angular_z,imu_gyro_z,imu_acc_x,imu_acc_y,imu_acc_z,cmd_linear_x,cmd_angular_z
0,2026-04-19 12:51:19.372790,241.810776,392.652985,317.574799,459.851196,0.858160,1,0.468,0.4700,1.053551,...,-0.039742,0.999210,0.000000,0.000057,0.000000,0.075417,-0.152032,9.657611,0.0,0.0
1,2026-04-19 12:51:19.791893,241.828949,392.772705,317.452179,460.556854,0.856332,1,0.468,0.4690,1.053551,...,-0.039742,0.999210,0.000000,0.000057,0.000000,0.003591,-0.138864,9.652822,0.0,0.0
2,2026-04-19 12:51:20.213423,241.728073,393.071808,317.712769,459.878265,0.858904,1,0.468,0.4690,1.053551,...,-0.039742,0.999210,0.000000,0.000057,0.000000,0.003591,-0.138864,9.652822,0.0,0.0
3,2026-04-19 12:51:20.611336,241.915802,392.829193,317.447937,460.025787,0.859890,1,0.468,0.4690,1.053551,...,-0.039742,0.999210,0.000000,0.000057,0.000000,0.003591,-0.138864,9.652822,0.0,0.0
4,2026-04-19 12:51:21.000654,241.638290,392.797302,317.639679,459.786499,0.857698,1,0.468,0.4700,1.053551,...,-0.039719,0.999211,0.000000,0.000291,0.000000,0.003591,-0.138864,9.652822,0.0,0.0
5,2026-04-19 12:51:21.406890,241.949799,393.323792,317.492065,460.246307,0.857491,1,0.468,0.4690,1.053551,...,-0.039719,0.999211,0.000000,0.000291,0.000000,0.061651,-0.183755,9.679159,0.0,0.0
6,2026-04-19 12:51:21.787283,241.876648,392.897217,317.587952,459.616058,0.859522,1,0.468,0.4700,1.053551,...,-0.039719,0.999211,0.000000,0.000291,0.000000,0.061651,-0.183755,9.679159,0.0,0.0
7,2026-04-19 12:51:22.191220,241.875107,393.187073,317.783295,459.316620,0.857167,1,0.468,0.4690,1.053551,...,-0.039719,0.999211,0.000000,0.000291,0.000000,0.061651,-0.183755,9.679159,0.0,0.0
8,2026-04-19 12:51:22.590316,241.629105,393.059937,317.531433,459.602173,0.858231,1,0.468,0.4690,1.053551,...,-0.039717,0.999211,0.000000,-0.000086,0.000000,0.061651,-0.183755,9.679159,0.0,0.0
9,2026-04-19 12:51:22.992773,241.972107,393.153198,317.863647,459.130127,0.858028,1,0.468,0.4690,1.053551,...,-0.039717,0.999211,0.000000,-0.000086,0.000000,0.037110,-0.175974,9.706692,0.0,0.0


In [87]:
distance_low = 40
distance_high = 45

low = distance_low/100
high = distance_high/100

In [88]:
import pandas as pd
import numpy as np

# =========================
# 1. 숫자형 변환
# =========================
numeric_cols = [
    "bbox_x1", "bbox_y1", "bbox_x2", "bbox_y2",
    "conf", "class",
    "center_m", "median_m",
    "pose_x", "pose_y",
    "linear_x", "angular_z",
    "imu_gyro_z",
    "imu_acc_x", "imu_acc_y", "imu_acc_z",
    "cmd_linear_x", "cmd_angular_z"
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")


# =========================
# 2. bbox 기본 feature
# =========================
df["bbox_width"] = df["bbox_x2"] - df["bbox_x1"]
df["bbox_height"] = df["bbox_y2"] - df["bbox_y1"]
df["bbox_area"] = df["bbox_width"] * df["bbox_height"]
df["bbox_center_x"] = (df["bbox_x1"] + df["bbox_x2"]) / 2.0
df["bbox_center_y"] = (df["bbox_y1"] + df["bbox_y2"]) / 2.0
df["bbox_diag"] = np.sqrt(df["bbox_width"]**2 + df["bbox_height"]**2)
df["bbox_aspect_ratio"] = df["bbox_width"] / df["bbox_height"].replace(0, np.nan)

# 잘못된 bbox 제거
df = df[df["bbox_width"] > 0]
df = df[df["bbox_height"] > 0]


# =========================
# 3. 거리 범위 필터
# =========================
df = df[df["median_m"].between(low, high)]


# =========================
# 4. 불필요 컬럼 제거
# =========================
drop_cols = [
    "timestamp",
    "pose_x", "pose_y",
    "ori_x", "ori_y", "ori_z", "ori_w",
    "linear_x", "center_m"   # 실제로 안 쓸 거면 제거
]

df = df.drop(columns=[c for c in drop_cols if c in df.columns], errors="ignore")


# =========================
# 5. class별 실제 물체 크기 매핑
# =========================
size_map = {
    1: (5.3, 3.3, 8.3),   # tank
    0: (1.1, 3.7, 0.5)    # soldier
}

df["obj_w_cm"] = np.nan
df["obj_h_cm"] = np.nan
df["obj_d_cm"] = np.nan

for cls, (w, h, d) in size_map.items():
    df.loc[df["class"] == cls, ["obj_w_cm", "obj_h_cm", "obj_d_cm"]] = [w, h, d]


# =========================
# 6. 실제 크기 기반 feature
# =========================
df["obj_area_cm2"] = df["obj_w_cm"] * df["obj_h_cm"]
df["obj_diag_cm"] = np.sqrt(df["obj_w_cm"]**2 + df["obj_h_cm"]**2)
df["obj_aspect_ratio"] = df["obj_w_cm"] / df["obj_h_cm"].replace(0, np.nan)


# =========================
# 7. depth 추정 핵심 ratio feature
# =========================
df["ratio_w"] = df["obj_w_cm"] / df["bbox_width"]
df["ratio_h"] = df["obj_h_cm"] / df["bbox_height"]
df["ratio_mean"] = (df["ratio_w"] + df["ratio_h"]) / 2.0

df["ratio_area"] = df["obj_area_cm2"] / df["bbox_area"]
df["ratio_diag"] = df["obj_diag_cm"] / df["bbox_diag"]

# 실제 물체 비율과 bbox 비율 차이
df["ratio_diff"] = abs(df["obj_aspect_ratio"] - df["bbox_aspect_ratio"])


# =========================
# 8. 로그 feature (선택)
# =========================
eps = 1e-6

df["log_bbox_width"] = np.log(df["bbox_width"] + eps)
df["log_bbox_height"] = np.log(df["bbox_height"] + eps)
df["log_bbox_area"] = np.log(df["bbox_area"] + eps)

df["log_ratio_w"] = np.log(df["ratio_w"] + eps)
df["log_ratio_h"] = np.log(df["ratio_h"] + eps)
df["log_ratio_mean"] = np.log(df["ratio_mean"] + eps)
df["log_ratio_area"] = np.log(df["ratio_area"] + eps)
df["log_ratio_diag"] = np.log(df["ratio_diag"] + eps)



In [ ]:
feature_cols = [
    "bbox_width",
    "bbox_height",
    "bbox_center_x",
    "bbox_center_y",
    "ratio_mean",
    "class",
    "bbox_y2",
    "bbox_y1" 
]

df["bbox_width"] = df["bbox_x2"] - df["bbox_x1"]
df["bbox_height"] = df["bbox_y2"] - df["bbox_y1"]
df["bbox_center_x"] = (df["bbox_x1"] + df["bbox_x2"]) / 2.0
df["bbox_center_y"] = (df["bbox_y1"] + df["bbox_y2"]) / 2.0

df["ratio_w"] = df["obj_w_cm"] / df["bbox_width"]
df["ratio_h"] = df["obj_h_cm"] / df["bbox_height"]
df["ratio_mean"] = (df["ratio_w"] + df["ratio_h"]) / 2.0

In [ ]:
df.shape
df.head()

,bbox_x1,bbox_y1,bbox_x2,bbox_y2,conf,class,median_m,angular_z,imu_gyro_z,imu_acc_x,...,ratio_diag,ratio_diff,log_bbox_width,log_bbox_height,log_bbox_area,log_ratio_w,log_ratio_h,log_ratio_mean,log_ratio_area,log_ratio_diag
31,124.523018,400.331543,209.014786,468.455963,0.867793,1,0.416,-0.255589,-0.195813,-0.016161,...,0.057524,0.365804,4.436654,4.221336,8.657990,-2.768931,-3.027393,-2.889835,-5.796032,-2.855529
34,62.138123,401.928528,153.166382,472.971680,0.871183,1,0.449,0.363329,0.259665,0.000000,...,0.054070,0.324751,4.511170,4.263287,8.774457,-2.843446,-3.069343,-2.950030,-5.912458,-2.917466
35,152.385162,399.397369,232.950943,462.295135,0.837430,1,0.415,0.363329,0.304361,-0.097564,...,0.061084,0.325160,4.389074,4.141511,8.530585,-2.721352,-2.947569,-2.828077,-5.668666,-2.795493


In [94]:
import os
import re
import glob
import numpy as np
import pandas as pd

INPUT_DIR = "."
OUTPUT_DIR = "./preprocessed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

size_map = {
    1: (5.3, 3.3, 8.3),   # tank
    0: (1.1, 3.7, 0.5)    # soldier
}

numeric_cols = [
    "bbox_x1", "bbox_y1", "bbox_x2", "bbox_y2",
    "conf", "class",
    "center_m", "median_m",
    "pose_x", "pose_y",
    "linear_x", "angular_z",
    "imu_gyro_z",
    "imu_acc_x", "imu_acc_y", "imu_acc_z",
    "cmd_linear_x", "cmd_angular_z"
]


def parse_range_from_filename(filename: str):
    """
    예:
    SoldierFront40-45.csv -> (40, 45)
    TankSide95-105.csv -> (95, 105)
    SoldierFront120-130Minus10.csv -> (120, 130)
    """
    base = os.path.splitext(os.path.basename(filename))[0]

    m = re.search(r'(\d+)-(\d+)', base)
    if not m:
        raise ValueError(f"파일명에서 거리 범위를 찾지 못함: {filename}")

    low = float(m.group(1)) / 100.0
    high = float(m.group(2)) / 100.0
    return low, high


def preprocess_csv(df: pd.DataFrame, low: float, high: float, size_map: dict) -> pd.DataFrame:
    df = df.copy()

    # 1. 숫자형 변환
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    required_cols = ["bbox_x1", "bbox_y1", "bbox_x2", "bbox_y2", "class", "median_m"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"필수 컬럼 없음: {missing}")

    # 2. bbox feature
    df["bbox_width"] = df["bbox_x2"] - df["bbox_x1"]
    df["bbox_height"] = df["bbox_y2"] - df["bbox_y1"]
    df["bbox_area"] = df["bbox_width"] * df["bbox_height"]
    df["bbox_center_x"] = (df["bbox_x1"] + df["bbox_x2"]) / 2.0
    df["bbox_center_y"] = (df["bbox_y1"] + df["bbox_y2"]) / 2.0
    df["bbox_diag"] = np.sqrt(df["bbox_width"]**2 + df["bbox_height"]**2)
    df["bbox_aspect_ratio"] = df["bbox_width"] / df["bbox_height"].replace(0, np.nan)

    df = df[df["bbox_width"] > 0]
    df = df[df["bbox_height"] > 0]

    # 3. 거리 구간 필터
    df = df[df["median_m"].between(low, high)]

    # 4. 불필요 컬럼 제거
    drop_cols = [
        "timestamp",
        "pose_x", "pose_y",
        "ori_x", "ori_y", "ori_z", "ori_w",
        "linear_x", "center_m"
    ]
    df = df.drop(columns=[c for c in drop_cols if c in df.columns], errors="ignore")

    # 5. 클래스별 실제 크기
    df["obj_w_cm"] = np.nan
    df["obj_h_cm"] = np.nan
    df["obj_d_cm"] = np.nan

    for cls, (w, h, d) in size_map.items():
        df.loc[df["class"] == cls, ["obj_w_cm", "obj_h_cm", "obj_d_cm"]] = [w, h, d]

    df = df.dropna(subset=["obj_w_cm", "obj_h_cm", "obj_d_cm"])

    # 6. 실제 크기 feature
    df["obj_area_cm2"] = df["obj_w_cm"] * df["obj_h_cm"]
    df["obj_diag_cm"] = np.sqrt(df["obj_w_cm"]**2 + df["obj_h_cm"]**2)
    df["obj_aspect_ratio"] = df["obj_w_cm"] / df["obj_h_cm"].replace(0, np.nan)

    # 7. ratio feature
    df["ratio_w"] = df["obj_w_cm"] / df["bbox_width"]
    df["ratio_h"] = df["obj_h_cm"] / df["bbox_height"]
    df["ratio_mean"] = (df["ratio_w"] + df["ratio_h"]) / 2.0
    df["ratio_area"] = df["obj_area_cm2"] / df["bbox_area"]
    df["ratio_diag"] = df["obj_diag_cm"] / df["bbox_diag"]
    df["ratio_diff"] = (df["obj_aspect_ratio"] - df["bbox_aspect_ratio"]).abs()

    # 8. log feature
    eps = 1e-6
    df["log_bbox_width"] = np.log(df["bbox_width"] + eps)
    df["log_bbox_height"] = np.log(df["bbox_height"] + eps)
    df["log_bbox_area"] = np.log(df["bbox_area"] + eps)

    df["log_ratio_w"] = np.log(df["ratio_w"] + eps)
    df["log_ratio_h"] = np.log(df["ratio_h"] + eps)
    df["log_ratio_mean"] = np.log(df["ratio_mean"] + eps)
    df["log_ratio_area"] = np.log(df["ratio_area"] + eps)
    df["log_ratio_diag"] = np.log(df["ratio_diag"] + eps)

    # 9. 정리
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.dropna()

    return df

csv_files = sorted(glob.glob(os.path.join(INPUT_DIR, "*.csv")))

print(f"처리 대상 파일 수: {len(csv_files)}")

for csv_path in csv_files:
    try:
        low, high = parse_range_from_filename(csv_path)
        print(f"[범위] {os.path.basename(csv_path)} -> low={low}, high={high}")

        df = pd.read_csv(csv_path)
        filename = os.path.basename(csv_path)
        is_minus = "Minus" in filename
        if is_minus:
            df["median_m"] = pd.to_numeric(df["median_m"], errors="coerce") - 0.1

        df_processed = preprocess_csv(df, low=low, high=high, size_map=size_map)

        save_name = f"preprocessed_{os.path.basename(csv_path)}"
        save_path = os.path.join(OUTPUT_DIR, save_name)
        df_processed.to_csv(save_path, index=False)

        print(f"[완료] {save_name} / rows={len(df_processed)}")

    except Exception as e:
        print(f"[실패] {csv_path}: {e}")



처리 대상 파일 수: 28
[범위] SoldierFront100-110.csv -> low=1.0, high=1.1
[완료] preprocessed_SoldierFront100-110.csv / rows=75
[범위] SoldierFront110-120.csv -> low=1.1, high=1.2
[완료] preprocessed_SoldierFront110-120.csv / rows=61
[범위] SoldierFront120-130Minus10.csv -> low=1.2, high=1.3
[완료] preprocessed_SoldierFront120-130Minus10.csv / rows=83
[범위] SoldierFront40-45.csv -> low=0.4, high=0.45
[완료] preprocessed_SoldierFront40-45.csv / rows=7
[범위] SoldierFront46-51.csv -> low=0.46, high=0.51
[완료] preprocessed_SoldierFront46-51.csv / rows=19
[범위] SoldierFront57-63.csv -> low=0.57, high=0.63
[완료] preprocessed_SoldierFront57-63.csv / rows=14
[범위] SoldierFront62-68.csv -> low=0.62, high=0.68
[완료] preprocessed_SoldierFront62-68.csv / rows=12
[범위] SoldierFront75-80.csv -> low=0.75, high=0.8
[완료] preprocessed_SoldierFront75-80.csv / rows=8
[범위] SoldierFront86-95.csv -> low=0.86, high=0.95
[완료] preprocessed_SoldierFront86-95.csv / rows=59
[범위] TankFrontSide105-110.csv -> low=1.05, high=1.1
[완료] preprocessed

In [93]:
len(os.listdir('.'))

30